# Cohort analysis: do the recurring groups keep meeting week after week?

**Where this fits.** `episodes_v1` → transactions → frequent itemsets (`apriori_from_scratch.ipynb`) → association rules (`association_rules.ipynb`) → **this notebook** → `outputs/tables/recurring_cohorts.csv`.

**The question.** The proposal asks which groups of people keep meeting each other, week after week. Support only says a group shared at least 200 distinct episodes. It does not say *when*. The 200 episodes could sit in one busy week or one lecture slot.

**A cohort** is a group of two or more students that is frequent at minsup 200 and *closed* (no bigger group has exactly the same support).

**What we measure for each cohort** (its episodes are the episodes that contain all of its students):

- how many days (of 28) and weeks (of 4) it appears in, and how its support splits between the first and second half;
- how regular its times are (most common weekday and start hour) and how much of it happens at night;
- how big its episodes are, and whether it still reaches minsup without the very large episodes;
- how many *sessions* its episodes make up (episodes less than one hour apart merged).

**Time conventions.** Day = `start_time // 86400` (day 0 is a Sunday, dataset paper p. 1), week = day // 7, weekday = day % 7 with 0 = Sunday. The clock hour assumes timestamp 0 is midnight, which the paper does not state, so hours and night share are descriptions only, never a filter.

**What this notebook does not show.** Recurring over days and weeks is still co-presence, not a social tie. Whether a cohort recurs more than the schedule explains is the null-model test, and the CSV is its input.

## Step 1. Load the episodes

Imports, paths and the settings used everywhere. `MINSUP = 200` is the count of episodes we chose earlier. We load `episodes_v1`, sort it by `episode_id`, and turn it into transactions (one set of students per episode) and a True/False table (one row per episode, one column per student).

In [ ]:
from itertools import combinations
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from mlxtend.frequent_patterns import apriori as mlx_apriori

PROJECT_ROOT = Path.cwd().parent                   # notebooks/ is one level below the project root
PROCESSED = PROJECT_ROOT / "data" / "processed"
TABLES = PROJECT_ROOT / "outputs" / "tables"

sys.path.append(str(PROJECT_ROOT))                 # so "from src..." works inside notebooks/
from src.config import load_config
from src.features.transactions import build_transactions, support_count, to_one_hot

config = load_config(PROJECT_ROOT / "config" / "config.yaml")
SLOT = config["time_parameters"]["slot_duration_sec"]    # 300 seconds
MINSUP = 200                                             # count of episodes, the chosen minimum support

episodes = pd.read_parquet(PROCESSED / "episodes_v1.parquet").sort_values("episode_id").reset_index(drop=True)
transactions = build_transactions(episodes)        # same order as the rows of `episodes`
n_episodes = len(transactions)
onehot = to_one_hot(transactions)                  # True where the student is in the episode
print(n_episodes, "episodes,", onehot.shape[1], "students")

## Step 2. Frequent itemsets at minsup 200

Same step as in `association_rules.ipynb`: mlxtend `apriori` finds every group of students that shares at least 200 episodes. mlxtend returns fractions, so we use `(MINSUP - 0.5) / N` (just below 200 / N, to avoid rounding errors) and turn the result back into counts. `supports` is a dictionary: group of students (sorted tuple) → number of episodes.

In [ ]:
share = (MINSUP - 0.5) / n_episodes
frequent = mlx_apriori(onehot, min_support=share, use_colnames=True, low_memory=True)

supports = {}                                      # {sorted tuple of student ids: support count}
for itemset, support in zip(frequent["itemsets"], frequent["support"]):
    key = tuple(sorted(int(item) for item in itemset))
    supports[key] = round(support * n_episodes)    # fraction back to a count

print(len(supports), "frequent itemsets at minsup", MINSUP)

## Step 3. Cohorts: the closed groups of two or more students

A group is **not closed** if a bigger frequent group (one more student) has the same support: then every episode of the small group also contains that extra student, so the small group adds nothing new. We mark those groups, keep the closed ones, and keep only groups with at least two students. We expect 735 closed itemsets and 343 cohorts, as in `apriori_from_scratch.ipynb`.

In [ ]:
not_closed = set()
for itemset, support in supports.items():
    if len(itemset) < 2:
        continue
    for smaller in combinations(itemset, len(itemset) - 1):    # the group without one student
        if supports[smaller] == support:                       # same support: the smaller group is not closed
            not_closed.add(smaller)

closed = {}
for itemset, support in supports.items():
    if itemset not in not_closed:
        closed[itemset] = support

cohorts = {}                                       # closed groups of two or more students
for itemset, support in closed.items():
    if len(itemset) >= 2:
        cohorts[itemset] = support

print(len(closed), "closed itemsets,", len(cohorts), "cohorts. Not closed:", sorted(not_closed))
assert len(closed) == 735 and len(cohorts) == 343

## Step 4. Describe every episode

We add a few columns to each episode: the day and week it starts, the weekday, how many students it has, whether it is mostly at night (`night_share` of at least 0.5), and whether it is one of the **5% largest episodes** (large = at least the 95th percentile of students per episode). The large episodes matter because one episode with many students contains a huge number of small groups.

In [ ]:
episodes["day"] = episodes["start_time"] // 86400          # day 0 .. 27
episodes["week"] = episodes["day"] // 7                    # week 0 .. 3
episodes["weekday"] = episodes["day"] % 7                  # 0 = Sunday
episodes["n_students"] = [len(t) for t in transactions]
episodes["is_night"] = episodes["night_share"] >= 0.5      # mostly between 00:00 and 07:00

large_threshold = episodes["n_students"].quantile(0.95)
episodes["is_large"] = episodes["n_students"] >= large_threshold

print("large episode = at least", large_threshold, "students;", int(episodes["is_large"].sum()), "large episodes,",
      int(episodes["is_night"].sum()), "night episodes")

## Step 5. Two small helper functions

- `count_sessions` counts how many separate sessions a list of episodes makes up. Episodes are sorted by start time. A new session starts when the next episode begins more than one hour after everything before it has ended. If one long get-together was split into many episodes, they count as one session.
- `lift_range` gives the lowest and highest lift over all ways to split a cohort into A and B (lift = N × σ(A∪B) / (σ(A) × σ(B))).

In [ ]:
def count_sessions(rows, max_gap_sec):
    rows = rows.sort_values("start_time")
    sessions = 0
    last_end = None
    for start, end in zip(rows["start_time"], rows["end_time"] + SLOT):    # end_time is the start of the last slot
        if last_end is None or start - last_end > max_gap_sec:
            sessions += 1                                                  # a gap: a new session starts
            last_end = end
        else:
            last_end = max(last_end, end)                                  # same session goes on
    return sessions


def lift_range(cohort, support, supports, n_total):
    lifts = []
    for size in range(1, len(cohort)):
        for antecedent in combinations(cohort, size):
            consequent = tuple(item for item in cohort if item not in antecedent)
            lifts.append(n_total * support / (supports[antecedent] * supports[consequent]))
    return min(lifts), max(lifts)

## Step 6. One row per cohort

For each cohort we pick its episodes (the rows where every student of the cohort is present), check that their number equals the support, and measure everything listed at the top. The result is `table`, sorted by support. The first rows are the cohorts with the most episodes.

In [ ]:
rows = []
for cohort, support in cohorts.items():
    mask = onehot[list(cohort)].all(axis=1)                # episodes that contain every student of the cohort
    mine = episodes[mask]
    assert len(mine) == support                            # recount of the support
    low, high = lift_range(cohort, support, supports, n_episodes)
    rows.append({
        "members": " ".join(str(student) for student in cohort),
        "size": len(cohort),
        "support": support,
        "days_seen": mine["day"].nunique(),
        "weeks_seen": mine["week"].nunique(),
        "support_first_half": int((mine["day"] < 14).sum()),
        "support_second_half": int((mine["day"] >= 14).sum()),
        "top_weekday_share": mine["weekday"].value_counts().iloc[0] / support,
        "top_hour_share": mine["start_hour"].value_counts().iloc[0] / support,
        "night_episode_share": mine["is_night"].mean(),
        "night_share_mean": mine["night_share"].mean(),
        "mean_episode_students": mine["n_students"].mean(),
        "large_episode_share": mine["is_large"].mean(),
        "support_without_large": int((~mine["is_large"]).sum()),
        "survives_without_large": int((~mine["is_large"]).sum()) >= MINSUP,
        "sessions_1h": count_sessions(mine, 3600),
        "min_lift": low,
        "max_lift": high,
    })

table = pd.DataFrame(rows).sort_values(["support", "members"], ascending=[False, True]).reset_index(drop=True)
print(len(table), "cohorts; by size:", table["size"].value_counts().sort_index().to_dict(),
      "| support from", table["support"].min(), "to", table["support"].max(), ", median", table["support"].median())
table.head(10).round(3)

## Step 7. Second check of the supports

The loop above already compared the number of episodes with the support. Here every cohort support is counted once more with `support_count` from `src/features/transactions.py`, a different piece of code. Nothing is printed if all 343 agree.

In [ ]:
for cohort, support in cohorts.items():
    assert support_count(transactions, cohort) == support
print("all", len(cohorts), "supports confirmed")

## Step 8. Do the cohorts recur week after week?

We count in how many weeks (of 4) each cohort appears, list the cohorts that miss a week, and look at the number of days seen and at the split between the first and second half of the 28 days (days 0–13 and 14–27).

In [ ]:
print("Cohorts by number of weeks (of 4) in which they appear")
display(table["weeks_seen"].value_counts().sort_index().to_frame("cohorts"))
display(table[table["weeks_seen"] < 4][["members", "support", "days_seen", "weeks_seen", "support_first_half", "support_second_half"]])

print("Days seen (of 28), minimum / quartiles / maximum:", table["days_seen"].quantile([0, 0.25, 0.5, 0.75, 1]).tolist())
both_halves = (table["support_first_half"] > 0) & (table["support_second_half"] > 0)
print("In both halves of the 28 days:", int(both_halves.sum()), "of", len(table))
first_share = table["support_first_half"] / table["support"]
print("Share of the support in the first half, minimum / quartiles / maximum:", first_share.quantile([0, 0.25, 0.5, 0.75, 1]).round(2).tolist())

### Plots: days seen and the two halves

Left: each dot is a cohort, with its support on the x axis and the number of days it is seen on the y axis. Right: its support in the first half against the second half; dots on the grey line have the same support in both halves.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].scatter(table["support"], table["days_seen"], s=14, alpha=0.5, color="#2a78d6")
axes[0].set_xlabel("support (episodes)")
axes[0].set_ylabel("days seen (of 28)")
axes[0].set_title("Support against days seen")
axes[0].grid(alpha=0.3)

axes[1].scatter(table["support_first_half"], table["support_second_half"], s=14, alpha=0.5, color="#2a78d6")
top = max(table["support_first_half"].max(), table["support_second_half"].max())
axes[1].plot([0, top], [0, top], color="#898781", lw=1)    # same support in both halves
axes[1].set_xlabel("support in days 0-13")
axes[1].set_ylabel("support in days 14-27")
axes[1].set_title("First half against second half")
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Step 9. Regular times and night

`top_weekday_share` is the share of a cohort's episodes on its most common weekday (the baseline for no pattern is 1/7 = 0.14, although the largest of seven shares is above that by chance). `top_hour_share` is the same for the start hour (baseline 1/24 = 0.04). We also count the cohorts with at least half of their episodes at night, and show the ten cohorts with the highest support.

In [ ]:
print("top weekday share, minimum / quartiles / maximum:", table["top_weekday_share"].quantile([0, 0.25, 0.5, 0.75, 1]).round(2).tolist())
print("top hour share, minimum / quartiles / maximum:", table["top_hour_share"].quantile([0, 0.25, 0.5, 0.75, 1]).round(2).tolist())
print("cohorts with at least half of their episodes at night:", int((table["night_episode_share"] >= 0.5).sum()), "of", len(table))
table.head(10)[["members", "size", "support", "days_seen", "weeks_seen", "night_episode_share", "top_hour_share"]].round(3)

## Step 10. How much do the large episodes matter?

A cohort that exists only inside very large episodes (for example 40 students together) tells us little about people meeting each other. For each cohort we count how many of its episodes are large, and how many cohorts fall below minsup 200 when the large episodes are removed.

In [ ]:
print("cohorts with at least half of their episodes large:", int((table["large_episode_share"] >= 0.5).sum()), "of", len(table))
below = table["support_without_large"] < MINSUP
print("cohorts below minsup", MINSUP, "without the large episodes:", int(below.sum()), "of", len(table))
print("share of a cohort's episodes that are large, minimum / quartiles / maximum:",
      table["large_episode_share"].quantile([0, 0.25, 0.5, 0.75, 1]).round(2).tolist())

## Step 11. One session or many episodes?

The linking can split one get-together into several episodes (when a member leaves, the overlap drops below 0.7). To see how much, `sessions_1h` merges a cohort's episodes that are less than one hour apart. If a cohort has 200 episodes but only 20 sessions, support overstates the number of separate meetings. The one hour is a diagnostic choice, not a tuned parameter.

In [ ]:
print("sessions per cohort, median:", table["sessions_1h"].median(), "| maximum:", table["sessions_1h"].max())
print("episodes per session, median:", round((table["support"] / table["sessions_1h"]).median(), 1))
print("sessions per day seen, median:", round((table["sessions_1h"] / table["days_seen"]).median(), 2))

## Step 12. The cohorts that do not depend on large episodes

`survives_without_large` is True when a cohort still has at least 200 episodes after the large episodes are removed. These are the cohorts closest to small recurring groups. We count them, look at their size and weeks, and compare the typical episode size with the other cohorts.

In [ ]:
robust = table[table["survives_without_large"]]
others = table[~table["survives_without_large"]]
print(len(robust), "cohorts survive; by size:", robust["size"].value_counts().sort_index().to_dict())
print("weeks seen:", robust["weeks_seen"].value_counts().sort_index().to_dict(), "| mostly at night:", int((robust["night_episode_share"] >= 0.5).sum()))
print("median students per episode:", round(robust["mean_episode_students"].median(), 1), "for these,",
      round(others["mean_episode_students"].median(), 1), "for the others")
robust.head(10)[["members", "size", "support", "support_without_large", "days_seen", "night_episode_share", "sessions_1h"]].round(3)

## Step 13. Write the file for the null models

One row per cohort, with the settings that produced it (`episodes_v1`, minsup, θ, gap, RSSI threshold) so the file can always be traced back. Shares are rounded to four decimals. No cohort is dropped here: a stricter filter is still an open decision, and `survives_without_large` lets us apply one later.

In [ ]:
out = table.copy()
out["episodes_version"] = "episodes_v1"
out["minsup"] = MINSUP
out["jaccard_threshold"] = config["episode_parameters"]["jaccard_threshold"]
out["max_gap_slots"] = config["episode_parameters"]["max_gap_slots"]
out["rssi_threshold"] = config["bluetooth_parameters"]["rssi_threshold"]
for column in ["top_weekday_share", "top_hour_share", "night_episode_share", "night_share_mean",
               "mean_episode_students", "large_episode_share", "min_lift", "max_lift"]:
    out[column] = out[column].round(4)

TABLES.mkdir(parents=True, exist_ok=True)
out.to_csv(TABLES / "recurring_cohorts.csv", index=False)

check = pd.read_csv(TABLES / "recurring_cohorts.csv")
assert len(check) == 343 and list(check.columns) == list(out.columns)
print("wrote", TABLES / "recurring_cohorts.csv", check.shape)

### Findings (`episodes_v1`, minsup 200)

- **343 cohorts** (closed itemsets of two or more students): 181 pairs, 90 trios, 55 groups of four, 16 of five and one of six. Support runs from 200 to 523 episodes (median 226).
- **Week after week.** 340 of the 343 cohorts have at least one episode in each of the four weeks; three (243 648, 242 276, 359 700) appear in three weeks and mostly in the second half. All 343 appear in both halves of the 28 days. The median cohort is seen on 19 of the 28 days (quartiles 17 to 20, minimum 11), and its support is split about evenly between the halves (median share in the first half 0.48). So a cohort's support is not one busy week.
- **Support is not the number of meetings.** Merging a cohort's episodes that are less than one hour apart gives a median of 20 sessions (at most 75) for supports of 200 to 523, about 11.6 episodes per session and about one session per day seen. Minsup 200 therefore asks for far fewer separate sessions than it sounds. Not tested why: the linking may split one session when the members change (Jaccard below 0.7), or the group may really meet in short bursts. The one hour is a diagnostic choice, not a parameter of the analysis.
- **Most cohorts rest on large episodes.** 241 of 343 cohorts have at least half of their episodes among the 5% largest episodes (26 students or more, 1,132 episodes), and 296 fall below minsup 200 without them. Only 47 survive (41 pairs and 6 trios); they appear in all four weeks and their episodes have a median of 5.1 students, against 41.2 for the other cohorts. Two students being in the same large episode says little about meeting each other, so the 47 are the cohorts closest to small recurring groups. The column `survives_without_large` marks them in the CSV.
- **Regularity.** The share of a cohort's episodes on its most common weekday has median 0.30 (baseline 1/7 = 0.14, but the largest of seven shares is above that by chance alone), and the share on its most common start hour has median 0.22 (baseline 1/24 = 0.04). The cohorts (292 309) and (292 427) have no night episodes, about 34 students per episode, 36% and 28% of their episodes on one weekday and about 20% in one start hour. That looks like large events tied to a timetable; there is no timetable in the data, so it stays a hypothesis.
- **Night.** 19 cohorts have at least half of their episodes at night, and 18 of them are among the 47 that survive without large episodes. The four cohorts with the highest support, (47 449), (448 449), (47 448) and (47 448 449), have 438 to 523 episodes on 24 to 25 days, about 45% to 49% of them at night, and about 5 students per episode. This is consistent with a group that is almost always together, possibly shared housing; the data cannot show that.
- **For the question "which groups keep meeting each other, week after week".** Almost every cohort does recur in every week and in both halves, so recurrence in time holds. It does not by itself mean meeting by choice or by obligation (not tested here), and most of the 343 are co-presence in large episodes. Whether a cohort recurs more than the schedule and each student's own activity explain is the null-model test, and `recurring_cohorts.csv` is its input.
- **Limits.** Weeks are 7-day blocks from timestamp 0 and an episode belongs to the day it starts. Hours and the night share assume timestamp 0 is midnight (not stated in the paper). The large-episode threshold (26 students, the 95th percentile of episode size) and the one-hour session gap are choices. All results are for RSSI −90, θ 0.7, gap 1.